# Suite VLGPRT — Printing Verilog

The printer writes one fixed layout and the parentheses precedence needs; printing validates the tree and checks
the standard first.

In [ ]:
import { Errors, type Syntax as F } from "@mbse/programs/Framework";
import { SystemVerilog2023, Syntax as S, Verilog2005 } from "@mbse/programs/Verilog";

import { equal, raises } from "./support.js";

const P = (node: F.SyntaxNode) => SystemVerilog2023.print(node);
const name = (spelling: string) => new S.Identifier({ spelling });
const ref = (spelling: string) => new S.NameExpression({ name: name(spelling) });
const number = (spelling = "1") => new S.IntegerLiteral({ spelling });
const binary = (left: any, operator: string, right: any) => new S.BinaryExpression({ left, operator, right });
const [a, b, c] = [ref("a"), ref("b"), ref("c")];

/** Text reads and prints back to itself. */
function roundtrip(text: string): string {
  const printed = P(SystemVerilog2023.parse(text));
  equal(printed, text);
  return printed;
}

## VLGPRT-01 · Parentheses follow IEEE 1800's precedence, and right-associative operators

In [ ]:
equal(P(binary(binary(a, "+", b), "*", c)), "(a + b) * c");
equal(P(binary(a, "-", binary(b, "-", c))), "a - (b - c)");
equal(P(binary(binary(a, "-", b), "-", c)), "a - b - c");
equal(P(binary(a, "->", binary(b, "->", c))), "a -> b -> c");
equal(P(binary(binary(a, "->", b), "->", c)), "(a -> b) -> c");
const cond = () => new S.ConditionalExpression({ condition: ref("a"), consequence: ref("b"), alternative: ref("c") });
equal(P(new S.ConditionalExpression({ condition: cond(), consequence: ref("b"), alternative: cond() })),
  "(a ? b : c) ? b : a ? b : c");
equal(P(new S.UnaryExpression({ operator: "-", operand: binary(a, "+", b) })), "-(a + b)");
equal(P(new S.UnaryExpression({ operator: "~", operand: new S.UnaryExpression({ operator: "-", operand: a }) })), "~(-a)");
equal(P(new S.IndexExpression({ value: binary(a, "+", b), index: number() })), "(a + b)[1]");
equal(P(new S.InsideExpression({ value: binary(a, "==", b), set: [number()] })), "(a == b) inside {1}");
equal(P(binary(a, "+", new S.InsideExpression({ value: b, set: [c] }))), "a + (b inside {c})");
equal(P(binary(new S.AssignmentExpression({ target: a, operator: "=", value: b }), "+", c)), "(a = b) + c");
equal(P(new S.IncrementExpression({ operator: "++", operand: binary(a, "+", b) })), "++(a + b)");
equal(P(new S.CastExpression({ type: binary(a, "+", b), value: c })), "(a + b)'(c)");
const clocked = (sequence: any) => new S.ClockedSequence({ clock: new S.EventControl({ events: [new S.EventExpression({
  expression: ref("k") })] }), sequence });
const always = (operand: any) => new S.UnaryProperty({ operator: "always", operand });
const step = (delay: any, sequence: any) => new S.DelayStep({ delay, sequence });
equal(P(new S.BinaryProperty({ left: always(a), operator: "and", right: b })), "(always a) and b");
equal(P(new S.BinaryProperty({ left: new S.BinaryProperty({ left: a, operator: "and", right: always(b) }), operator: "or", right: c })),
  "a and (always b) or c");
equal(P(new S.BinaryProperty({ left: a, operator: "and", right: always(b) })), "a and always b"); // nothing follows it
equal(P(new S.DelaySequence({ first: new S.BinarySequence({ left: a, operator: "or", right: b }), steps: [step(number(), c)] })),
  "(a or b) ##1 c");
equal(P(new S.RepetitionSequence({ sequence: new S.DelaySequence({ first: a, steps: [step(number(), b)] }), operator: "*",
  count: number("2") })), "(a ##1 b)[*2]");
equal(P(new S.ConditionalProperty({ condition: a, consequence: always(b), alternative: c })), "if (a) (always b) else c");
equal(P(new S.BinarySequence({ left: a, operator: "throughout", right: new S.BinarySequence({ left: b, operator: "throughout",
  right: c }) })), "a throughout b throughout c");
equal(P(new S.BinarySequence({ left: new S.BinarySequence({ left: a, operator: "throughout", right: b }), operator: "throughout",
  right: c })), "(a throughout b) throughout c");
equal(P(new S.UnaryProperty({ operator: "not", operand: new S.BinaryProperty({ left: a, operator: "and", right: b }) })),
  "not (a and b)");
equal(P(new S.DelaySequence({ steps: [step(binary(a, "+", number()), b), step(number("2"), clocked(c))] })),
  "##(a + 1) b ##2 @(k) c");
equal(P(new S.DelaySequence({ steps: [step(number("2"), clocked(c)), step(number(), ref("d"))] })), "##2 (@(k) c) ##1 d");
equal(P(new S.DelaySequence({ first: clocked(a), steps: [step(number(), b)] })), "(@(k) a) ##1 b");
equal(P(new S.ImplicationProperty({ antecedent: a, operator: "|->", consequent: new S.ImplicationProperty({ antecedent: b,
  operator: "|=>", consequent: c }) })), "a |-> b |=> c");
equal(P(new S.TimedStatement({ timing: new S.CycleDelay({ value: binary(a, "+", number()) }) })), "##(a + 1);");
console.log("ok");

## VLGPRT-02 · Every expression and part prints alone

In [ ]:
const cases: [F.SyntaxNode, string][] = [
  [new S.MemberExpression({ value: a, member: name("m") }), "a.m"],
  [new S.RangeSelect({ value: a, left: number("7"), operator: ":", right: number("0") }), "a[7:0]"],
  [new S.RangeSelect({ value: a, left: number("0"), operator: "+:", right: number("4") }), "a[0 +: 4]"],
  [new S.RealLiteral({ spelling: "1.5" }), "1.5"], [new S.TimeLiteral({ spelling: "1ns" }), "1ns"],
  [new S.UnbasedUnsizedLiteral({ value: "z" }), "'z"], [new S.StringLiteral({ text: "hi" }), '"hi"'],
  [new S.Concatenation({ items: [a, number()] }), "{a, 1}"], [new S.Replication({ count: number("2"), items: [a] }), "{2{a}}"],
  [new S.AssignmentPattern({ items: [new S.PatternItem({ value: number() }), new S.PatternItem({ key: a, value: number("2") }),
    number("3")] }), "'{default: 1, a: 2, 3}"],
  [new S.AssignmentPattern({ type: new S.NamedType({ name: name("t") }), items: [number()] }), "t'{1}"],
  [new S.CallExpression({ callee: a, arguments: [number(), new S.NamedConnection({ name: name("b"), value: number("2") })] }),
    "a(1, .b(2))"],
  [new S.SystemCall({ name: "$bits", arguments: [new S.IntegerAtomType({ keyword: "int" })] }), "$bits(int)"],
  [new S.SystemCall({ name: "$time" }), "$time"],
  [new S.CastExpression({ type: new S.ImplicitType({ signing: "unsigned" }), value: a }), "unsigned'(a)"],
  [new S.MacroUsage({ name: name("M"), arguments: "1, 2" }), "`M(1, 2)"], [new S.MacroUsage({ name: name("W") }), "`W"],
  [new S.DollarExpression(), "$"],
  [new S.NameExpression({ name: new S.ScopedName({ scope: name("p"), name: name("x") }) }), "p::x"],
  [new S.RangeDimension({ left: number("3"), right: number("0") }), "[3:0]"], [new S.UnsizedDimension(), "[]"],
  [new S.AssociativeDimension(), "[*]"], [new S.QueueDimension({ bound: number("4") }), "[$:4]"], [new S.QueueDimension(), "[$]"],
  [new S.IntegerVectorType({ keyword: "bit", signing: "signed", dimensions: [new S.SizeDimension({ size: number("4") })] }),
    "bit signed [4]"],
  [new S.ImplicitType({ dimensions: [new S.RangeDimension({ left: number("1"), right: number("0") })] }), "[1:0]"],
  [new S.EnumType({ base: new S.IntegerAtomType({ keyword: "int" }), members: [new S.EnumMember({ name: name("A"), value: number() })] }),
    "enum int {A = 1}"],
  [new S.StructType({ keyword: "union", members: [new S.StructMember({ type: new S.KeywordType({ keyword: "string" }),
    declarators: [new S.VariableDeclarator({ name: name("s") })] })] }), "union { string s; }"],
  [new S.AnsiPort({ name: name("p") }), "p"], [new S.InterfacePort({ name: name("i") }), "interface i"],
  [new S.PortReference({ name: name("r") }), "r"],
  [new S.IncludeDirective({ path: "types.svh", system: true }), "`include <types.svh>"],
  [new S.TimedStatement({ timing: new S.DelayControl({ value: binary(a, "+", number()) }) }), "#(a + 1);"],
  [new S.VirtualInterfaceType({ interface: name("bus"), parameters: [number("4")], modport: name("mp") }), "virtual bus #(4).mp"],
  [new S.NewExpression({ scope: new S.ScopedName({ scope: name("p"), name: new S.ParameterizedName({ name: name("c"),
    parameters: [number()] }) }), arguments: [a] }), "p::c #(1)::new(a)"],
  [new S.NewCopyExpression({ value: new S.CallExpression({ callee: a }) }), "new a()"],
  [new S.NewCopyExpression({ value: binary(a, "+", number()) }), "new (a + 1)"],
  [new S.ForwardTypedefDeclaration({ name: name("t") }), "typedef t;"],
  [new S.ConstraintPrototype({ static: true, name: name("k") }), "static constraint k;"],
  [new S.NameExpression({ name: new S.LocalName({ name: name("x") }) }), "local::x"],
  [new S.RandomizeWithExpression({ call: new S.CallExpression({ callee: ref("randomize") }) }), "randomize() with {}"],
  [new S.DistExpression({ value: a, items: [new S.DistItem({ value: number() })] }), "a dist {1}"],
  [new S.DefaultSkew({ input: new S.ClockingSkew({ edge: "posedge" }) }), "default input posedge;"],
  [new S.DefaultSkew({ output: new S.ClockingSkew({ delay: new S.DelayControl({ value: number("2") }) }) }), "default output #2;"],
  [new S.BinsSelection({ keyword: "bins", name: name("s"), select: new S.BinaryBinsSelect({ left: a, operator: "&&",
    right: new S.BinaryBinsSelect({ left: b, operator: "||", right: c }) }) }), "bins s = a && (b || c);"],
  [new S.BinsSelection({ keyword: "bins", name: name("s"), select: new S.NotBinsSelect({ operand: new S.BinaryBinsSelect({
    left: a, operator: "&&", right: b }) }) }), "bins s = !(a && b);"],
  [new S.BinsSelection({ keyword: "bins", name: name("s"), select: new S.FilteredBinsSelect({ select: new S.BinaryBinsSelect({
    left: a, operator: "&&", right: b }), filter: c }) }), "bins s = (a && b) with (c);"],
  [new S.BinsSelection({ keyword: "bins", name: name("s"), select: new S.FilteredBinsSelect({
    select: new S.NotBinsSelect({ operand: a }), filter: c }) }), "bins s = !a with (c);"],
  [new S.SystemCall({ name: "$time", attributes: [new S.AttributeInstance({ specs: [new S.AttributeSpec({ name: name("now") })] })] }),
    "$time (* now *) ()"],
];
for (const [node, text] of cases) equal(P(node), text);
console.log("ok");

## VLGPRT-03 · Layout: blocks open on their header's line, `end else`, one item per line

In [ ]:
roundtrip(`module m;
    always_ff @(posedge clk) begin
        if (a) begin
            b <= 1;
        end else if (c) begin
            b <= 2;
        end else begin
            b <= 3;
        end
        if (a)
            b <= 4;
        else if (c)
            b <= 5;
        else
            b <= 6;
        unique case (a) inside
            1, [2:3]: b <= 1;
            4: begin
                b <= 2;
            end
            default: ;
        endcase
        for (int i = 0; i < 2; i++)
            a = i;
        do begin
        end while (a);
        do
            a = 1;
        while (a);
        wait (a);
        wait (a) b = 1;
        wait (a) begin
        end
        #1 a = 1;
        @(posedge clk) begin
        end
        assert (a);
        assert (a) b = 1; else b = 2;
        assert (a) begin
        end else begin
        end
        assert (a) else $error("x");
    end
endmodule
`);
roundtrip(`module m;
    if (W) begin : a
    end else if (V) begin : b
    end else
        assign x = 1;

    if (W)
        assign x = 2;
    else
        assign x = 3;

    case (W)
        1:
            assign x = 4;
        default: begin
        end
    endcase

    for (genvar i = 0; i < 2; i++) begin : g
    end : g

    for (j = 0; j < 2; j = j + 1)
        assign x = 5;
endmodule
`);
console.log("ok");

## VLGPRT-04 · Layout: design units, declarations and directives, with blank lines around what spans lines

In [ ]:
roundtrip(`// lead
\`timescale 1ns / 1ps

/* the package */
package p;
    typedef struct packed {
        logic a;
        logic b;
    } s;
    var struct packed {
        logic a;
        logic b;
    } v;
    typedef union { int i; } u;

    function automatic int f(input int a = 1, output b);
        return a;
    endfunction : f

    task t;
        input a;
    endtask
endpackage : p

module m (a, b);
    input wire a;
    output reg b;

\`ifdef X
    wire x;
\`elsif Y
    wire y;
\`else
    wire z;
\`endif
endmodule

interface i
    import p::*;
#(
    parameter int W = 1,
    parameter type T = logic
) (
    input logic clk
);
    modport mp (input clk);
endinterface

program q #(
    localparam N = 1
);
endprogram
`);
const text = P(SystemVerilog2023.parse("// lead\n`timescale 1ns / 1ps\n/* the package */\npackage p; typedef struct packed { logic a; logic b; } s; endpackage\n"));
equal(text, "// lead\n`timescale 1ns / 1ps\n\n/* the package */\npackage p;\n    typedef struct packed {\n        logic a;\n        logic b;\n    } s;\nendpackage\n");
console.log("ok");

## VLGPRT-05 · Printing validates the tree and checks the standard

In [ ]:
raises(Errors.PrintError, () => P(new S.BinaryExpression({ operator: "+", right: number() })), "a BinaryExpression needs a left");
raises(Errors.PrintError, () => Verilog2005.print(new S.PackageDeclaration({ name: name("p") })),
  "PackageDeclaration is not Verilog, but this is Verilog-2005");
equal(Verilog2005.print(new S.ModuleDeclaration({ keyword: "module", name: name("m") })), "module m;\nendmodule");
equal(P(new S.Comment({ text: " c" })), "// c");
equal(P(new S.Comment({ block: true, text: " c " })), "/* c */");
console.log("ok");

## VLGPRT-06 · Layout: trailing comments, conditionals among statements, bodies that span lines

In [ ]:
roundtrip(`module m;  // trailing
    wire a;  /* after a */

    initial begin
        x = (a inside {1, 2}) && b;

\`ifdef A
        x = 1;
\`else
        x = 2;
\`endif

        case (a)
            1:
                if (b)
                    x = 1;
                else
                    x = 2;
            default: ;
        endcase
        assert (a) $display("pass");
        assert (a) begin
            x = 1;
        end else $error("fail");
        ;
    end
endmodule
`);
console.log("ok");

## VLGPRT-07 · Layout: every option of headers, ports, declarations and statements

In [ ]:
roundtrip(`\`define EMPTY

module a;
endmodule  // after a

// before b
module b
    import p::*;
(x, y);
    input var logic x;
    output y;
endmodule

module c
    import p::*;
(
    output var logic z = 1'b0,
    w
);
    wire #3 n;
    wire #(d + 1) m;
    const var static int k = 1;
    var v;
    static struct {
        logic a;
        logic b;
    } s;
    struct {
        logic a;
        logic b;
    } u;
    const struct packed {
        logic a;
        logic b;
    } [1:0] t = '0;
    typedef struct packed { logic a; } one;
    typedef enum {A} e;

    /* two
       lines */
    function int f(input var int i, j);
        return 1;
    endfunction

    task r;
        return;
    endtask

    initial begin
        ->> ev;
        disable named;
        disable fork;
        unique if (a)
            b = 1;
        for (; ; );
        for (int k = 0, m = 1; ; );
        #(a + 1) b = 1;
    end
endmodule
`);
console.log("ok");

## VLGPRT-08 · Layout: classes, their parameters, bases, members and prototypes

In [ ]:
roundtrip(`typedef class driver;

virtual class base #(
    type T = int,
    int N = 2
) extends parent #(T) implements i1, i2;
    local const static int count = 0;
    extern virtual function void f(int x);
    pure virtual task t;

    function new(int x = 0);
        super.new(x);
    endfunction
endclass : base

class derived extends base #(byte, 3)(1, .x(2));
endclass

interface class i1 extends j, k;
    pure virtual function void h;
endclass

module m #(
    int W = 8,
    type U = logic
);
    wire struct packed {
        logic a;
        logic b;
    } w;
endmodule
`);
console.log("ok");

## VLGPRT-09 · Layout: constraints, inline constraints and `randcase`

In [ ]:
roundtrip(`class c;
    rand int q[];

    constraint k {  // first
        x > 0;
        soft y < 10;
        x -> {
            y == 1;
        }
        x ->
            y == 2;
        if (x)
            y > 2;
        else if (z) {
            y < 3;
        } else
            y == 4;
        foreach (q[i]) {
            q[i] < 5;
        }
        solve x before y, q;
        disable soft y;
        unique {x, [1:2]};
        x dist {1 := 2, [3:4] :/ 5, 7, default :/ 1};

\`ifdef SIM
        x < 9;
\`endif
    }

    extern static constraint e;

    function void f;
        ok = randomize() with { // inline
x -> y; if (a) { b; } else c; if (d) e; foreach (q[i]) q[i] > 0; solve a before b; disable soft x; unique {a, b}; };
        ok = obj.randomize(x) with (x) {};
        ok = randomize() with {
\`ifdef SIM
x > 1;
\`endif
y < 3; };
        r = q.find(item) with (item > 1);
        randcase
            1: a = 1;
            2: begin
            end
        endcase
        void'(obj.randomize());
    endfunction
endclass

constraint c::e {
    x != y;
}
`);
console.log("ok");

## VLGPRT-10 · Layout: clocking blocks, property and sequence declarations, assertions and `let`

In [ ]:
roundtrip(`module m;
    default clocking cb @(posedge clk);
        default input #1step output #0;
        input #1 a, b;
        output negedge c;
        inout d = top.x;
        input #2 output #3 e;
    endclocking : cb

    global clocking gc @(posedge fast);
    endclocking

    default clocking @(negedge clk);
        property inner;
            if (a) b;
        endproperty
    endclocking

    default clocking cb;
    default disable iff (rst);

    sequence s1(x, int y = 1, local input logic z);
        int v;
        @(posedge clk) (a, v = x) ##1 b[*2] ##[1:3] c[->1] ##[0:$] d[=2:3] ##[1:$] e[*0:$] ##1 f[*1:$] ##(y) g;
    endsequence : s1

    sequence s2;
        ##1 a ##2 (b ##1 c)[*3] and first_match(d ##1 e, v++) or f intersect g within h throughout i;
    endsequence

    property p1(sequence q, untyped u);
        disable iff (rst) q |-> not (a until b) and s_eventually [1:$] c;
    endproperty

    property p2;
        if (a) strong(b ##1 c) else weak(d) implies nexttime [2] e iff always [1:3] f;
    endproperty

    property p3;
        accept_on (x) a s_until b or sync_reject_on (y) c until_with d;
    endproperty

    property p4;
        case (m) 0, 1: a |=> b; default: (c #=# d); endcase;
    endproperty

    a1: assert property (@(posedge clk) disable iff (rst) a |=> b) else $error("x");
    cover sequence (s1(1));
    restrict property (a ##1 @(negedge clk) b);
    assume property (s_nexttime a or @(negedge clk) b |-> c);
    assert #0 (x);
    let max(a, b = 1) = a > b ? a : b;

    initial begin
        ##2;
        ##1 cb.c <= 1;
        cb.c <= ##2 a;
        chk: assert (x);
        a2: assert property (p1(s1, 1));
        expect (@(posedge clk) a ##1 b) else $error;
        lab: x = 1;
    end
endmodule
`);
console.log("ok");

## VLGPRT-11 · Layout: covergroups, coverpoints with their bins, and crosses

In [ ]:
roundtrip(`class c;
    covergroup cg(int lo, ref int r) @(posedge clk);
        option.per_instance = 1;
        type_option.weight = 2;

        cp_a: coverpoint a iff (en) {
            bins low[4] = {[0:3], 5} iff (x);
            bins mid = {[4:$]} with (item % 2 == 0);
            wildcard bins w = {4'b1??0};
            illegal_bins bad = default;
            ignore_bins ig[] = (1 => 2 => 3), (4 => 5[*2:3] => 6[->1] => 7[=2]);
            bins tr = (1, 2 => 3);
            bins d = default sequence;
            option.at_least = 2;
            bins s = cp_b with (item > 1);
            bins e = a + 1;
        }

        coverpoint b;
        cp_c: coverpoint a + b;
        bit [3:0] cp_t: coverpoint a;

        x_ab: cross cp_a, b iff (en) {
            bins both = binsof(cp_a.low) && binsof(b) intersect {[1:2]};
            ignore_bins no = !binsof(cp_a) || binsof(cp_a.mid);
            bins w = x_ab with (cp_a > 1);
            bins p = (binsof(b));

            function int f;
            endfunction
        }

        cross cp_a, cp_c;
    endgroup : cg

    covergroup cg2 with function sample(int v);
        coverpoint v;
    endgroup

    covergroup cg3;
    endgroup
endclass
`);
console.log("ok");

## VLGPRT-12 · Layout: attributes start their item's or statement's line, and sit after their operator

In [ ]:
roundtrip(`(* top, version = "1.0" *) module m (
    (* keep *) input a,
    output b
);
    (* keep *) (* dont_touch = 1 *) wire w;

    (* full_case, parallel_case *) initial
        (* lab = 2 *) case (a)
            default: ;
        endcase

    assign b = a + (* mark *) w ? (* x *) a : b;

    initial begin
        x = f (* inline *) (1);
        y = $clog2 (* fast *) (8);
        z = - (* neg *) a;
        i (* inc *)++;
        ++ (* pre *) i;
        lab: (* attr *) x = 1;
    end

    class c;
        (* keep *) int p;
    endclass
endmodule
`);
console.log("ok");

## VLGPRT-13 · Layout: delays, streaming concatenations, repeated patterns, tolerance ranges, empty arguments, `$root`, `$unit` and `type()`

In [ ]:
roundtrip(`module m;
    wire #(1, 2, 3) w;
    assign #(1:2:3, 4) w = x;
    wire #(5) v;

    initial begin
        x = {>>{a, b}};
        x = {<< 8 {a with [0 +: 2]}};
        x = {<< byte {a with [1]}};
        x = {>> 4 {b with [1:2]}};
        x = '{2{a, b}};
        x = (1:2:3);
        x = a inside {[1 +/- 2], [3 +%- 4], [5:6]};
        x = f(a, , b);
        $display(a, , b);
        x = $root.top.a;
        x = {};
        x = $unit::a;
        y = type(a)'(b);
        #(1:2:3) x = 1;
    end

    var type(a) v2;
    var type(int) v3;
    u i (a, , b);
endmodule
`);
const typical = () => new S.MinTypMaxExpression({ min: ref("a"), typ: ref("b"), max: ref("c") });
const delayedAssign = new S.ContinuousAssign({ delay: new S.DelayControl({ value: typical(), fall: number("2") }), assignments: [
  new S.AssignmentExpression({ target: ref("x"), operator: "=", value: binary(typical(), "+", ref("y")) })] });
equal(P(new S.SourceText({ items: [delayedAssign] })), "assign #(a:b:c, 2) x = (a:b:c) + y;\n");
console.log("ok");


## VLGPRT-14 · Layout: strengths, net types, aliases, `defparam`, time units, enumerations' ranges, randomized members, `const ref` and `ref static`, and type restrictions

In [ ]:
roundtrip(`package p;
    timeunit 1ns / 10ps;
    nettype logic [1:0] nt;
    nettype real total with p::sum;
    typedef enum {A[2] = 0, B[1:2], C} e;
    typedef struct { rand int a; } one;
    typedef struct {
        rand int a;
        randc bit [1:0] b;
    } two;

    function int f(const ref int a, ref static int b);
    endfunction
endpackage

module m #(
    parameter type enum T = e,
    type interface class I = i
);
    timeprecision 1ps;
    wire (strong0, weak1) w1;
    trireg (small) t;
    wire (highz1, pull0) vectored [1:0] w2;
    tri scalared [3:0] w3 = 1;
    assign (pull1, supply0) #1 a = b;
    alias a = b = c[1:0];
    defparam u.W = 2, v.X = 3;
    wire (supply0, supply1) struct packed {
        logic a;
        logic b;
    } pair;
endmodule
`);
console.log("ok");


## VLGPRT-15 · Layout: non-ANSI and explicit ports, and modports' subroutines, clocking blocks and explicit ports

In [ ]:
roundtrip(`module a (.p(x), .q(), .r({y, z}), , w[1:0], {u, v}, x[0], w[0 +: 1]);
    input [1:0] x, w;
    input y, z, u, v;
endmodule

module b (
    input .p(x + 1),
    .q(y),
    output logic z,
    .r()
);
endmodule

interface i;
    modport mp (import f, import g, export h, import task t(int a), import function int k, clocking cb, input .a(b), input c, output .d());
endinterface
`);
console.log("ok");


## VLGPRT-16 · Layout: specifiers, default arguments, empty items, interfaces' types, skipped `foreach` variables and `ifdef conditions

In [ ]:
roundtrip(`class :final c extends b(default);
    ;
    local typedef int t;
    protected typedef struct {
        int a;
        int b;
    } s;

    virtual function :initial :final void f;
    endfunction

    extern virtual function :extends void g;

    virtual task :final t2;
    endtask

    constraint :initial k {
        x > 0;
    }

    extern constraint :extends :final k2;

    constraint k3 {
        foreach (q[, j, , k])
            q[j] > 0;
    }
endclass

\`ifdef (A && !B || (C -> D) || E <-> F)
module m; endmodule
\`elsif (G)
module g; endmodule
\`endif

module n (
    bus_if bus
);
    typedef bus.data_t t;

    initial
        foreach (q[i, ])
            x = 1;
endmodule
`);
console.log("ok");


## VLGPRT-17 · Layout: procedural continuous assignments, `wait fork`, `wait_order`, timed triggers, repeated event controls and elaboration tasks

In [ ]:
roundtrip(`module m;
    $error("bad %d", 1);
    $info;
    $fatal(1, , "x");

    generate
        if (1)
            $warning("w");
    endgenerate

    initial begin
        force a[0] = 1;
        release a;
        assign b = 1;
        deassign b;
        wait fork;
        wait_order (a, b.c) x = 1; else y = 1;
        wait_order (a, b);
        wait_order (a) else begin
            y = 1;
            z = 2;
        end
        ->> #1 e;
        ->> @(posedge clk) e;
        ->> e;
        a <= repeat (2) @(posedge clk) b;
    end
endmodule
`);
console.log("ok");


## VLGPRT-18 · Layout: DPI imports and exports, `bind`, `extern` design units and `.*` ports

In [ ]:
roundtrip(`module m;
    import "DPI-C" function int f(int a);
    import "DPI-C" context task t;
    import "DPI-C" pure c_name = function int g;
    import "DPI" function void h;
    export "DPI-C" function f;
    export "DPI-C" e = task t;
    bind n mon k ();
endmodule

bind m checker_unit c (.*);
bind m: u1, u2 mon #(1) i (.a(b)), j ();
bind top.m mon k ();

extern module e #(
    parameter W = 1
) (
    input a
);

extern interface ei (
    input a
);

extern program ep;

extern module en (a, b);

module e (.*);
endmodule
`);
console.log("ok");


## VLGPRT-19 · Layout: a production on a line, but for code that spans lines

In [ ]:
roundtrip(`module m;
    initial
        randsequence (main)
            main : first second(2) | third := 2 | rand join (0.5) first third | rand join second(1) first := 1 { x = 3; };
            first : {
                x = 1;  // one
            };
            second(int n = 1) : if (a) third else first | repeat (2) third | case (b) 0, 1: first; default: second(3); endcase | if (c) first;
            third : {
                x = 2;
                if (y) begin
                    z = 1;
                end
            } first;
            void f : { return; };
            int g : { x = 1; return 1; };
        endsequence

    initial
        randsequence ()
            a : b;
            b : { };
        endsequence
endmodule
`);
console.log("ok");


## VLGPRT-20 · Layout: checkers, their instances, package exports and nets of user-defined types

In [ ]:
roundtrip(`package p;
    checker pc(a);
        assert property (a);
    endchecker

    export q::*;
    export q::x, r::y;
    export *::*;
endpackage

checker c(input logic clk, a, output bit b = 0, property pr, sequence s = a ##1 a, untyped u, event e = $inferred_clock);
    default clocking @(posedge clk);
    endclocking

    rand bit r;
    rand const int k;
    bit v = 0;

    checker inner;
    endchecker
endchecker : c

module m;
    nettype logic nt;
    nt #1 w, v = 1;
    nt w2;
    c c1 (.clk(clk), .a(x));
    p::pc c4 (x);

    always @(posedge clk) begin
        c c2 (clk, x), c3 (.*);
        p::pc c5 (x);
    end
endmodule
`);
console.log("ok");


## VLGPRT-21 · Layout: tagged unions and values, `case matches`, predicates and patterns; a tagged value is parenthesized where it is an operand

In [ ]:
roundtrip(`module m;
    typedef union tagged packed {
        int a;
        void b;
    } t;
    typedef union soft {
        int a;
        bit [31:0] c;
    } s;

    initial begin
        x = tagged a 5;
        x = tagged b;
        f(tagged a (1 + 2), tagged a tagged b 1);
        y = (tagged a 5) + 1;
        case (x) matches
            tagged a .v &&& v > 0: ;
            tagged b: ;
            .*: ;
            default: ;
        endcase
        if (x matches tagged a .v &&& v > 1);
        y = x matches tagged a .v &&& v > 1 ? v : 0;
        case (p) matches
            '{.q, 2}: ;
            '{f: .r, g: tagged a .*}: ;
        endcase
        if (a &&& b);
    end
endmodule
`);
const taggedValue = new S.TaggedExpression({ member: name("a"), value: binary(ref("x"), "+", number()) });
equal(P(new S.SourceText({ items: [new S.ContinuousAssign({ assignments: [new S.AssignmentExpression({
  target: ref("y"), operator: "=", value: binary(taggedValue, "*", number("2")) })] })] })), "assign y = (tagged a (x + 1)) * 2;\n");
console.log("ok");


## VLGPRT-22 · Layout: user-defined primitives, their tables, and gate instances

In [ ]:
roundtrip(`primitive mux (output o, input s, a, b);
    table
        // s a b : o
        0 1 ? : 1;
        1 ? 0 : 0;
    endtable
endprimitive : mux

primitive latch (q, d, en);
    output q;
    reg q;
    input d, en;
    initial q = 1'b0;
    table
        (01) 1 : ? : 1;
        r ? : 0 : -;
        * 0 : ? : -;
        (0x) x : 1 : x;
    endtable
endprimitive

primitive dff (output reg q = 0, input d, clk);
    table
        0 r : ? : 0;
    endtable
endprimitive

module m;
    and #(1, 2) g1 (o, a, b), g2 (o2, a, b);
    nand (strong0, weak1) (o3, a, b);
    bufif0 #3 b1[1:0] (o4, a, en);
    pullup (p);
    pulldown (weak0) pd (q);
    tranif1 t1 (x, y, en);
    not (n1, n2, a);
    mux u1 (o, s, a, b);
    latch #1 (q, d, en);
endmodule
`);
console.log("ok");
